[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/seap-udea/MontuPython/blob/main/examples/MontuPython-HeliacalRises.ipynb)

<p align="left"><img src="https://github.com/seap-udea/MontuPython/raw/main/montu/data/montu-python-logo-complete.webp" width="300" /></p>

# Heliacal Rises of Sirius

In this notebook, we'll explore the tools provided by MontuPython to calculate the heliacal rise of celestial bodies. Specifically, we will compare the results of four different visibility models to calculate the heliacal rise of Sirius during the era of its first *apokatastasis* (Sothic cycle) around **2782 BCE**.

First, if you are running this script in Google Colab you need to install the package:

In [2]:
# %pip install -Uq montu

In [3]:
%matplotlib inline
import montu
import numpy as np
import pandas as pd

# Autoreload refreshes local copies of montu when you edit the source.
# Not available in Google Colab (Python 3.12+): IPython autoreload still imports removed module `imp`.
# Uncomment only in a local Jupyter notebook:
# %load_ext autoreload
# %autoreload 2

MontuPython version 0.21.6. 𓋹 𓍘 𓋴 𓎛 𓂡 𓁘 (ii-ti m Htp, HkAx Hn'-k)


## 1. Setting up the Scene

To calculate a heliacal rise, we need three things:
1. **An observer**: Where are we looking from? Let's use the ancient site of Thebes.
2. **A celestial body**: What are we looking for? Sirius, the brightest star in the night sky.
3. **A time window**: When are we looking? The summer of 2782 BCE (which corresponds to astronomical year -2781).

In [4]:
# 1. The Observer
site = montu.Observer(site='thebes')

# 2. The Celestial Body
sirius = montu.Stars(subset='bright', ProperName='Sirius')

# 3. The Time Window (Summer of 2782 BCE)
start_date = montu.Time('-2781-06-01', calendar='mixed')
end_date = montu.Time('-2781-08-01', calendar='mixed')

Loading stellar catalogue montu_stellar_catalogue_v38_bright.csv


We can use the high-level `montu.heliacal_rise()` wrapper to run searches, but we'll use the core `montu.HeliacalRise` class directly to dive into the details.

## 2. Ptolemy's Arcus Visionis (The Simplest Model)

The simplest and oldest method to calculate heliacal rises was popularized by Ptolemy. It is based on a single geometric concept: the **Arcus Visionis** ($AV$).

The Arcus Visionis is simply the depression angle of the Sun below the horizon at the exact moment the star crosses the eastern horizon (rises).
If the Sun is far enough below the horizon, the sky will be dark enough for the star to be seen just as it rises.

**The Equation:**
The star is considered visible if:
$$\text{AV}_{calc} \ge \text{AV}_{crit}$$
where $\text{AV}_{calc} = -h_{\odot}$ (the altitude of the Sun is negative, so its depression is positive), and $\text{AV}_{crit}$ is a critical value that depends on the brightness of the object. For a first-magnitude star like Sirius, Ptolemy used $\text{AV}_{crit} \approx 15^\circ$.

In [5]:
# Set up the model
model_ptolemy = montu.HeliacalRise(model='ptolemy')

# Compute the rise
result_ptolemy = model_ptolemy.compute(sirius, site, start_date, end_date)

# Print the results
model_ptolemy.print_rises(result_ptolemy, body_label='Sirius')

ptolemy — 1 date(s)
  [1] -2781-07-19 00:00:00  03:46:49.156  -2781-06-26 00:00:00.000000  hrw 0-I-Mesut-5  Sirius -0.00°  Sun -15.09°
  source: Toomer, G. J. (1998). Ptolemy's Almagest. Princeton University Press. Book XIII, Chapter 7: "On the heliacal risings and settings of the planets".


## 3. Schaefer (1987) - Fixed Depression

Bradley Schaefer (1987) improved upon the ancient models by considering atmospheric extinction (how much light the atmosphere absorbs) and the limiting magnitude of the naked eye.

However, for a quick and robust calculation, he noted that for a specific location and star, the heliacal rise occurs when the Sun reaches a specific, fixed depression angle during morning twilight.

**The Equation:**
The model evaluates visibility at the instant the Sun reaches a specific depression angle (e.g., $h_{\odot} = -11^\circ$). The star is visible if its extinguished magnitude is brighter than the local limiting magnitude:

$$V + k \cdot X \le V_{lim}(zenith) - k \cdot (X - 1)$$

Where:
- $V$ is the catalog visual magnitude of the star.
- $k$ is the extinction coefficient (typically $0.25$ mag/airmass).
- $X$ is the airmass of the star ($X \approx \csc(h_\star)$).
- $V_{lim}(zenith)$ is the limiting magnitude at the zenith (typically $6.0$).

In [6]:
# We will use a standard solar depression of -11 degrees
model_schaefer87 = montu.HeliacalRise(model='schaefer1987', sun_depression=-11.0)

result_schaefer87 = model_schaefer87.compute(sirius, site, start_date, end_date)
model_schaefer87.print_rises(result_schaefer87, body_label='Sirius')

schaefer1987 — 1 date(s)
  [1] -2781-07-19 00:00:00  04:09:01.712  -2781-06-26 00:00:00.000000  hrw 0-I-Mesut-5  Sirius 4.53°  Sun -11.00°
  source: Schaefer, B. E. (1987). "Heliacal rise phenomena". Journal for the History of Astronomy, 18(11), 19-33.


## 4. Schaefer (1985) - Twilight Scan

In 1985, Schaefer published a comprehensive BASIC algorithm to calculate heliacal rises from first principles. Instead of assuming a fixed time or sun depression, this model *scans* the entire morning twilight, minute by minute.

It calculates the sky brightness at the position of the star and determines if the contrast is sufficient for the human eye to detect it.

**The Equations:**
The sky brightness $B$ is a complex empirical function of the zenith distance of the star, the azimuth separation from the sun, and the solar altitude. The detection threshold flux $E_{th}$ is calculated from $B$, and the limiting magnitude is given by:

$$V_{lim} = -16.57 - k \cdot X - 2.5 \log_{10}(E_{th})$$

The star is visible if $V \le V_{lim}$ at any point during the twilight.

In [7]:
# The model will scan the twilight every 2 minutes
model_schaefer85 = montu.HeliacalRise(model='schaefer1985', step_minutes=2)

result_schaefer85 = model_schaefer85.compute(sirius, site, start_date, end_date)
model_schaefer85.print_rises(result_schaefer85, body_label='Sirius')

schaefer1985 — 1 date(s)
  [1] -2781-07-11 00:00:00  04:32:25.452  -2781-06-18 00:00:00.000000  hrw 0-IV-Shemu-27  Sirius 2.90°  Sun -5.96°
  source: Schaefer, B.E. 1985, Sky & Telescope 70, 261–263 (BASIC listing, lines 34–35 and 55–81).


## 5. Belokrylov et al. (2011) - Twilight Scan

A more recent model by Belokrylov et al. (2011) also scans the twilight, but uses simplified empirical equations fitted to observational data.

**The Equations:**
First, they correct the magnitude for atmospheric extinction:

$$m' = V + k(X - 1) + (k - k_0)$$

Where $k_0$ is a reference extinction ($0.25$). Then they find a limiting solar altitude $h_{lim}$ based on $m'$. For bright stars ($m' < 4.2$):

$$h_{lim} = -2.47 - 1.23 m'$$

Finally, a small correction $\Delta h$ is added based on the angular separation $\rho$ between the sun and the star:

$$h_{theor} = h_{lim} - 0.0338 \max(0, 58^\circ - \rho)$$

The star becomes visible when the actual solar altitude $h_{\odot} \le h_{theor}$.

In [8]:
model_belokrylov = montu.HeliacalRise(model='belokrylov2011')

result_belokrylov = model_belokrylov.compute(sirius, site, start_date, end_date)
model_belokrylov.print_rises(result_belokrylov, body_label='Sirius')

belokrylov2011 — 1 date(s)
  [1] -2781-07-11 00:00:00  04:26:25.450  -2781-06-18 00:00:00.000000  hrw 0-IV-Shemu-27  Sirius 1.67°  Sun -7.24°
  source: Belokrylov, R. O., Belokrylov, S. V., & Nickiforov, M. G. (2011). "Model of the stellar visibility during twilight". Bulgarian Astronomical Journal, 16, 50-72.


## 6. Under the Hood: Exploring the Internal Mechanics

MontuPython's `HeliacalRise` class hides a lot of complexity. But how does it actually compute these events internally? The general algorithm iterates over each day in your time window. For each day, it checks if the visibility criteria are met during the morning twilight.

To do this, it leverages core MontuPython methods like `montu.Sun.when_is_twilight()`, `montu.Sun.conditions_in_sky()`, and `where_in_sky()`. Let's manually replicate the inner workings of each model for a single day: **July 25, 2782 BCE** (the day Ptolemy's model predicts the heliacal rise).

In [9]:
# Set up the specific day and the Sun object
day = montu.Time('-2781-07-25 00:00:00', calendar='mixed')
sun = montu.Sun()

### Manually reproducing Ptolemy's Arcus Visionis

Ptolemy evaluates the Sun's altitude precisely when the star rises. Internally, MontuPython does this analytically using spherical trigonometry to find the exact Local Sidereal Time (LST) of the rising branch.

We can approximate this logic by finding the exact rise time of Sirius using `conditions_in_sky()` and then evaluating the Sun's position.

In [10]:
# 1. Find when Sirius rises
# We can use the conditions_in_sky method we just implemented!
sirius.conditions_in_sky(day, site, inplace=True)
rise_time_jed = float(sirius.data.iloc[0].rise_time)
rise_time = montu.Time(rise_time_jed, format='jd')

print(f"Sirius rises at: {site.get_local_time(rise_time_jed)}")

# 2. Where is the Sun at that exact moment?
sun.where_in_sky(rise_time, site)
print(f"Sun altitude at Sirius rise: {sun.position.el:.2f} degrees")
print(f"(Notice how this closely matches the -15.09° Arcus Visionis calculated earlier!)")


Sirius rises at: 03:20:31.250
Sun altitude at Sirius rise: -20.19 degrees
(Notice how this closely matches the -15.09° Arcus Visionis calculated earlier!)


### Manually reproducing Schaefer (1987) fixed-depression

The fixed-depression model doesn't care when the star rises; it evaluates the sky exactly when the Sun is at a specific depression (e.g., $-11^\circ$). MontuPython uses `Sun.when_is_twilight()` to find this instant analytically without iterating.

In [23]:
# 1. Find when the Sun is at -11 degrees altitude
twilights = montu.Sun.when_is_twilight(day, site, sunbelow=-11.0)
morning_twilight_jed = min(twilights) # Morning is the earliest of the two intersections
morning_twilight = montu.Time(morning_twilight_jed, format='jd')

print(f"Sun reaches -11° at: {site.get_local_time(morning_twilight_jed)}")

# 2. Where is Sirius at this moment?
sirius.where_in_sky(morning_twilight, site)
print(f"Sirius altitude: {float(sirius.data.el[0]):.2f} degrees")

Sun reaches -11° at: 04:11:37.716
Sirius altitude: 8.44 degrees


### Manually reproducing the Twilight Scans (1985 & 2011)

The scanning algorithms (`schaefer1985` and `belokrylov2011`) are the most complex. They define a window between astronomical dawn (Sun at $-18^\circ$) and sunrise, and then advance step-by-step (e.g., every 2 minutes).

At each step, they update the positions of both the Sun and the star, and evaluate their complex empirical formulas.

In [32]:
# 1. Find the bounds of the scan
dawn_jed = min(montu.Sun.when_is_twilight(day, site, sunbelow=-18.0))
sun.conditions_in_sky(day, site)
sunrise_jed = sun.condition.rise_time

print(f"Scanning window: from {site.get_local_time(dawn_jed)} to {site.get_local_time(sunrise_jed)}\n")

# 2. Perform a 10-minute step scan manually
step_days = 10 / (24 * 60) # 10 minutes in days

for at_jed in np.arange(dawn_jed, sunrise_jed, step_days):
    at = montu.Time(at_jed, format='jd')
    
    # Update both body and sun
    sirius.where_in_sky(at, site)
    sun.where_in_sky(at, site)
    
    # Only care if the star is above the horizon
    if sirius.data.el[0] > 0:
        print(f"{site.get_local_time(at_jed)} | Sirius Alt: {sirius.data.el[0]:5.2f}° | Sun Alt: {sun.position.el:5.2f}°")
        # Internally, here is where MontuPython calls _schaefer1985_limiting_magnitude() 
        # or _belokrylov2011_threshold() to check if the star is visible at this instant.

Scanning window: from 03:33:11.527 to 05:03:33.075

03:33:11.527 | Sirius Alt:  8.44° | Sun Alt: -18.00°
03:43:11.523 | Sirius Alt:  8.44° | Sun Alt: -16.23°
03:53:11.528 | Sirius Alt:  8.44° | Sun Alt: -14.42°
04:03:11.524 | Sirius Alt:  8.44° | Sun Alt: -12.58°
04:13:11.529 | Sirius Alt:  8.44° | Sun Alt: -10.71°
04:23:11.525 | Sirius Alt:  8.44° | Sun Alt: -8.81°
04:33:11.530 | Sirius Alt:  8.44° | Sun Alt: -6.82°
04:43:11.526 | Sirius Alt:  8.44° | Sun Alt: -4.18°
04:53:11.531 | Sirius Alt:  8.44° | Sun Alt: -1.77°
05:03:11.527 | Sirius Alt:  8.44° | Sun Alt: -0.32°


## Conclusion

As we can see, all models point to late July 2782 BCE for the heliacal rise of Sirius from Thebes. The difference between the models is typically only a day or two.

The simpler models (Ptolemy and Schaefer 1987) are faster, but the twilight scan models (Schaefer 1985 and Belokrylov 2011) offer a more detailed physical simulation of the changing light conditions.